# 02 — Build and Run an Agent

This notebook walks you through creating an agent, running conversations,
and observing tool calls — all locally using the Microsoft Agent Framework.
Run cells top-to-bottom. Change `AGENT_NAME` below to work with any registered agent.

## Configuration

In [1]:
import sys, pathlib

# Ensure the local project root is first on sys.path so our `agents`
# package takes priority over the openai-agents site-package.
_project_root = str(pathlib.Path.cwd().parent)
if _project_root not in sys.path:
    sys.path.insert(0, _project_root)

# Change this to work with a different agent
AGENT_NAME = "code-helper"

## Install Dependencies

In [2]:
!uv sync

Resolved 226 packages in 4ms
Checked 221 packages in 1ms


## Connect and Create Agent

In [3]:
# List available knowledge bases in your Foundry project
# (Portal calls them "knowledge bases", the API lists them as "indexes")
import os
from dotenv import load_dotenv
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

load_dotenv("../.env")

# Support sovereign clouds (Azure Government, etc.)
authority = os.environ.get("AZURE_AUTHORITY_HOST")
cred_kwargs = {"authority": authority} if authority else {}
credential = DefaultAzureCredential(**cred_kwargs)

client_kwargs = {}
# Azure Government uses different credential scopes
if authority and "microsoftonline.us" in authority:
    client_kwargs["credential_scopes"] = ["https://cognitiveservices.azure.us/.default"]
# Disable SSL verification if configured (corporate proxy / SSL inspection)
if os.environ.get("DISABLE_SSL_VERIFY", "").lower() in ("true", "1", "yes"):
    client_kwargs["connection_verify"] = False

project = AIProjectClient(
    endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    credential=credential,
    **client_kwargs,
)

print("Knowledge bases in your Foundry project:\n")
for idx in project.indexes.list():
    index_name = getattr(idx, "index_name", "N/A")
    connection = getattr(idx, "connection_name", "N/A")
    print(f"  • Knowledge base:  {idx.name}")
    print(f"    Version:         {idx.version}")
    print(f"    Type:            {idx.type}")
    print(f"    Search index:    {index_name}")
    print(f"    Connection:      {connection}")
    print()

print("To use in .env, set: AZURE_AI_SEARCH_KNOWLEDGE_BASE=<knowledge base name above>")

Knowledge bases in your Foundry project:



/home/kiran/projects/agentframework-automation/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'ai-openai-arizona.openai.azure.us'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


ResourceNotFoundError: (404) Resource not found
Code: 404
Message: Resource not found

In [4]:
from dotenv import load_dotenv
import os

load_dotenv("../.env")

from agents.registry import REGISTRY
from agents._base.agent_factory import create_agent

entry = REGISTRY.get_agent(AGENT_NAME)
config = entry.config_class()
agent = create_agent(config)

print(f"✓ Agent '{config.agent_name}' assembled")
print(f"  Model:   {config.agent_deployment_name}")
print(f"  Endpoint: {config.azure_ai_project_endpoint[:50]}…")

✓ Agent 'code-helper' assembled
  Model:   gpt-5.1-2
  Endpoint: https://ai-openai-arizona.openai.azure.us/…


## Inspect Agent Tools

See what tools the agent has available. These are plain Python functions
that the Agent Framework auto-wraps and calls during conversations.

In [5]:
import importlib

module_name = AGENT_NAME.replace("-", "_")
try:
    tools_module = importlib.import_module(f"agents.{module_name}.tools")
    tools = getattr(tools_module, "TOOLS", [])
    if tools:
        print(f"Agent '{AGENT_NAME}' has {len(tools)} tool(s):")
        for tool in tools:
            name = tool.__name__ if hasattr(tool, "__name__") else str(tool)
            doc = (tool.__doc__ or "No description").strip().split("\n")[0]
            print(f"  • {name}: {doc}")
    else:
        print(f"Agent '{AGENT_NAME}' is instruction-only (no tools)")
except ModuleNotFoundError:
    print(f"No tools module found for '{AGENT_NAME}'")

Agent 'code-helper' has 1 tool(s):
  • greet_user: Greet a user by name.


## Send a Message

Run a single-turn conversation. The Agent Framework handles tool calling
automatically — you just send a prompt and get back a response.

In [6]:
# Change this to test different prompts
USER_MESSAGE = "Hello! What can you help me with?"

result = await agent.run(USER_MESSAGE)
response_text = result.text if hasattr(result, "text") else str(result)

print(f"User: {USER_MESSAGE}")
print(f"\nAgent: {response_text}")

/home/kiran/projects/agentframework-automation/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'ai-search-arizona.search.azure.us'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


User: Hello! What can you help me with?

Agent: I can help you with a range of questions, but only using the information in your organization’s knowledge base. Here’s what I can currently assist with, along with the sources I’d use:

1. **IT and Access**
   - IT support availability and hours  
     - Source: “question / answer – What are the IT support hours?”
   - How to reset your county password  
     - Source: “question / answer – How do I reset my county password?”

2. **Location / Office Info**
   - Address of Probation LA County headquarters  
     - Source: “question / answer – Where is the headquarters located?”

3. **Employee Policies (Probation LA County Employee Handbook)**
   - Time off / vacation policy (days per year, rollover rules)  
   - Telework / hybrid work policy (how many days, requirement for manager approval)  
   - Expense reimbursement process (portal used, deadline)  
     - Source: “Probation LA County Employee Handbook”

If your question is outside these

## Trigger a Tool Call

Send a prompt that should trigger the agent's greeting tool.
The Agent Framework calls the tool function automatically and
incorporates the result into the response.

In [7]:
# This should trigger the greet_user tool (for code-helper)
TOOL_MESSAGE = "Please greet Alice using your greeting tool."

result = await agent.run(TOOL_MESSAGE)
response_text = result.text if hasattr(result, "text") else str(result)

print(f"User: {TOOL_MESSAGE}")
print(f"\nAgent: {response_text}")

/home/kiran/projects/agentframework-automation/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'ai-search-arizona.search.azure.us'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


User: Please greet Alice using your greeting tool.

Agent: Hello, Alice! I'm the Code Helper agent. How can I assist you today with your coding or technical questions?


## Multi-Turn Conversation

Use a loop for an interactive multi-turn conversation.
Type `quit` or `exit` to stop.

In [8]:
from agents._base.agent_factory import agent_session

print(f"Chat with '{AGENT_NAME}' — type 'quit' to stop\n")

async with agent_session(config) as agent:
    while True:
        user_input = input("You: ")
        if user_input.strip().lower() in ("quit", "exit", ""):
            print("\n✓ Conversation ended")
            break

        result = await agent.run(user_input)
        response_text = result.text if hasattr(result, "text") else str(result)
        print(f"Agent: {response_text}\n")

Chat with 'code-helper' — type 'quit' to stop



/home/kiran/projects/agentframework-automation/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'ai-search-arizona.search.azure.us'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Agent: Hello! I'm the Code Helper agent. How can I help you with your code or project today?



/home/kiran/projects/agentframework-automation/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'ai-search-arizona.search.azure.us'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Agent: Probation LA County headquarters is located at **9150 E. Imperial Hwy, Downey, CA 90242**.  
(Source: *Probation LA County Employee Handbook*)


✓ Conversation ended


## Use the Programmatic API

The `agents` package provides a public API for scripting. The `run_agent()` function
is a sync convenience wrapper that creates the agent, runs the prompt, and returns
the response text.

In [9]:
from agents import run_agent

response = run_agent(config, "Explain what a Python decorator is in one sentence.")
print(f"Response: {response}")

/home/kiran/projects/agentframework-automation/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'ai-search-arizona.search.azure.us'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Response: The provided information does not mention Python or decorators, so I cannot answer this based on the available sources.


## Try a Different Agent

Switch to `doc-assistant` (or any other registered agent) and run a prompt.

In [10]:
other_agent = "doc-assistant"
other_entry = REGISTRY.get_agent(other_agent)
other_config = other_entry.config_class()
other = create_agent(other_config)

result = await other.run("What makes a good README?")
response_text = result.text if hasattr(result, "text") else str(result)
print(f"'{other_agent}' says:\n{response_text}")

/home/kiran/projects/agentframework-automation/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'ai-search-arizona.search.azure.us'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


'doc-assistant' says:
The available documents don’t define what makes a good README, so I can’t answer this from the provided knowledge base.

The sources I have cover:
- IT support hours, password reset, and headquarters location (source: “question answer” table).
- Time off, telework, and expense reimbursement policies (source: “Probation LA County Employee Handbook”).
- A description of the FieldTrack mobile reporting application (source: “FieldTrack is the mobile reporting application…”).

None of these describe README best practices, structure, or requirements.
